# AuraGateway vLLM CUDA 12.8 offline compatibility verifier v1


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from pathlib import Path

NOTEBOOK_NAME = "ag-vllm-cu128-offline-compatibility-v1"
INPUT_DIRECTORY_NAME = "auragateway_vllm_cu128_wheelhouse_v1"
OUTPUT_ZIP = Path("/kaggle/working/ag-vllm-cu128-offline-compatibility-v1.zip")
EVIDENCE_ROOT = Path("/kaggle/working/ag-vllm-cu128-offline-compatibility-v1")
VENV_ROOT = Path("/kaggle/working/auragateway_vllm_runtime_v1")
MAX_EXCERPT = 12000


def canonical_json(payload: object) -> str:
    return json.dumps(payload, ensure_ascii=True, separators=(",", ":"), sort_keys=True)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def sanitize(text: str) -> str:
    bounded = text[-MAX_EXCERPT:]
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
        os.environ.get("HOME", "<home>"): "<home>",
    }
    for source, replacement in replacements.items():
        if source:
            bounded = bounded.replace(source, replacement)
    return bounded


def run_probe(role: str, argv: list[str], *, timeout: float) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env={**os.environ, "PIP_DISABLE_PIP_VERSION_CHECK": "1"},
        )
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "timed_out": False,
            "status": "PASSED" if result.returncode == 0 else "FAILED",
            "stdout_excerpt": sanitize(result.stdout),
            "stderr_excerpt": sanitize(result.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": None,
            "timed_out": True,
            "status": "FAILED",
            "stdout_excerpt": sanitize(exc.stdout or ""),
            "stderr_excerpt": sanitize(exc.stderr or ""),
        }


if os.environ.get("AURAGATEWAY_CUSTOMER_DATA_PRESENT") == "1":
    raise RuntimeError("customer data is prohibited")

matches = tuple(Path("/kaggle/input").rglob(INPUT_DIRECTORY_NAME))
if len(matches) != 1 or not matches[0].is_dir() or matches[0].is_symlink():
    raise RuntimeError("expected exactly one governed wheelhouse directory")
input_root = matches[0]

if EVIDENCE_ROOT.exists():
    shutil.rmtree(EVIDENCE_ROOT)
EVIDENCE_ROOT.mkdir(parents=True)
if VENV_ROOT.exists():
    shutil.rmtree(VENV_ROOT)
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

required_files = {
    "requirements.in",
    "materialization.lock.txt",
    "requirements.lock.txt",
    "install_runtime.py",
    "runtime_manifest.json",
    "sha256_manifest.json",
    "materialization_receipt.json",
    "wheels",
}
if {path.name for path in input_root.iterdir()} != required_files:
    raise RuntimeError("wheelhouse top-level topology drifted")

installer_source = (input_root / "install_runtime.py").read_text(encoding="utf-8")
required_install_flags = ('"--no-index"', '"--find-links"', '"--require-hashes"')
if any(flag not in installer_source for flag in required_install_flags):
    raise RuntimeError("offline installer lost a required isolation flag")

sha_manifest = json.loads((input_root / "sha256_manifest.json").read_text(encoding="utf-8"))
entries = sha_manifest.get("entries")
if not isinstance(entries, list) or not entries:
    raise RuntimeError("wheelhouse SHA-256 manifest is invalid")
for entry in entries:
    if not isinstance(entry, dict):
        raise RuntimeError("wheelhouse SHA-256 entry is invalid")
    relative = Path(str(entry["path"]))
    if relative.is_absolute() or ".." in relative.parts:
        raise RuntimeError("wheelhouse SHA-256 path is unsafe")
    path = input_root / relative
    if not path.is_file() or path.is_symlink():
        raise RuntimeError("wheelhouse payload member is missing or unsafe")
    if sha256_file(path) != entry["sha256"]:
        raise RuntimeError("wheelhouse payload identity drifted")

runtime_manifest = json.loads((input_root / "runtime_manifest.json").read_text(encoding="utf-8"))
expected_runtime = {
    "python": "3.12",
    "cuda_variant": "cu128",
    "vllm_release": "0.19.1",
    "vllm": "0.19.1+cu128",
    "torch": "2.10.0+cu128",
    "torchaudio": "2.10.0+cu128",
    "torchvision": "0.25.0+cu128",
    "transformers": "5.5.3",
}
for key, value in expected_runtime.items():
    if runtime_manifest.get(key) != value:
        raise RuntimeError(f"runtime manifest drifted at {key}")

probe_records: list[dict[str, object]] = []
install = run_probe(
    "offline_isolated_install",
    [
        sys.executable,
        str(input_root / "install_runtime.py"),
        "--wheelhouse",
        str(input_root / "wheels"),
        "--lock",
        str(input_root / "requirements.lock.txt"),
        "--venv",
        str(VENV_ROOT),
    ],
    timeout=2400.0,
)
probe_records.append(install)
python = VENV_ROOT / "bin" / "python"

if install["status"] == "PASSED":
    probe_records.extend(
        [
            run_probe("pip_check", [str(python), "-m", "pip", "check"], timeout=120.0),
            run_probe(
                "gpu_topology",
                [
                    "nvidia-smi",
                    "--query-gpu=index,name,memory.total,compute_cap,driver_version",
                    "--format=csv,noheader,nounits",
                ],
                timeout=30.0,
            ),
            run_probe(
                "python_runtime",
                [str(python), "-c", "import platform; print(platform.python_version())"],
                timeout=30.0,
            ),
            run_probe(
                "torch_runtime",
                [
                    str(python),
                    "-c",
                    (
                        "import json,torch; print(json.dumps({"
                        "'torch':torch.__version__,'cuda':torch.version.cuda,"
                        "'available':torch.cuda.is_available(),"
                        "'device_count':torch.cuda.device_count()}))"
                    ),
                ],
                timeout=60.0,
            ),
            run_probe(
                "transformers_runtime",
                [
                    str(python),
                    "-c",
                    "import transformers; print(transformers.__version__)",
                ],
                timeout=60.0,
            ),
            run_probe(
                "vllm_distribution",
                [
                    str(python),
                    "-c",
                    "import importlib.metadata; print(importlib.metadata.version('vllm'))",
                ],
                timeout=60.0,
            ),
            run_probe(
                "vllm_module",
                [str(python), "-c", "import vllm; print(vllm.__version__)"],
                timeout=120.0,
            ),
            run_probe(
                "vllm_native_extension",
                [
                    str(python),
                    "-c",
                    "import importlib; importlib.import_module('vllm._C'); print('ok')",
                ],
                timeout=120.0,
            ),
        ]
    )

observed = {str(record["command_role"]): record for record in probe_records}


def reject_semantic(role: str, message: str) -> None:
    record = observed.get(role)
    if record is None:
        return
    record["status"] = "FAILED"
    record["semantic_error"] = message


if "gpu_topology" in observed and observed["gpu_topology"]["status"] == "PASSED":
    lines = [line.strip() for line in str(observed["gpu_topology"]["stdout_excerpt"]).splitlines()]
    if len(lines) != 2:
        reject_semantic("gpu_topology", "expected exactly two GPU records")
    else:
        for index, line in enumerate(lines):
            parts = [part.strip() for part in line.split(",")]
            if len(parts) != 5 or parts[0] != str(index):
                reject_semantic("gpu_topology", "GPU topology record is malformed")
                break
            if parts[1] not in {"Tesla T4", "NVIDIA T4"} or parts[3] != "7.5":
                reject_semantic("gpu_topology", "GPU topology is not two T4 devices")
                break

if "python_runtime" in observed and observed["python_runtime"]["status"] == "PASSED":
    if not str(observed["python_runtime"]["stdout_excerpt"]).strip().startswith("3.12."):
        reject_semantic("python_runtime", "Python runtime is not 3.12.x")

if "torch_runtime" in observed and observed["torch_runtime"]["status"] == "PASSED":
    try:
        torch_payload = json.loads(str(observed["torch_runtime"]["stdout_excerpt"]).strip())
    except json.JSONDecodeError:
        reject_semantic("torch_runtime", "Torch runtime output is not JSON")
    else:
        expected_torch = {
            "torch": "2.10.0+cu128",
            "cuda": "12.8",
            "available": True,
            "device_count": 2,
        }
        if torch_payload != expected_torch:
            reject_semantic("torch_runtime", "Torch or CUDA runtime identity drifted")

expected_stdout = {
    "transformers_runtime": "5.5.3",
    "vllm_distribution": "0.19.1+cu128",
    "vllm_module": "0.19.1+cu128",
    "vllm_native_extension": "ok",
}
for role, expected_value in expected_stdout.items():
    record = observed.get(role)
    if record is not None and record["status"] == "PASSED":
        if str(record["stdout_excerpt"]).strip() != expected_value:
            reject_semantic(role, f"expected {expected_value}")

for record in probe_records:
    role = str(record["command_role"])
    (EVIDENCE_ROOT / f"10_{role}.json").write_text(
        canonical_json(record) + "\n",
        encoding="utf-8",
    )

required_roles = {
    "offline_isolated_install",
    "pip_check",
    "gpu_topology",
    "python_runtime",
    "torch_runtime",
    "transformers_runtime",
    "vllm_distribution",
    "vllm_module",
    "vllm_native_extension",
}
failed = sorted(
    role
    for role in required_roles
    if role not in observed or observed[role]["status"] != "PASSED"
)

summary = {
    "schema_version": "1.0.0",
    "diagnostic_id": "auragateway-vllm-cu128-offline-compatibility-v1",
    "captured_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "status": "PASSED" if not failed else "FAILED",
    "failed_required_roles": failed,
    "input_runtime_manifest_sha256": sha256_file(input_root / "runtime_manifest.json"),
    "input_materialization_receipt_sha256": sha256_file(
        input_root / "materialization_receipt.json"
    ),
    "model_requests_performed": 0,
    "benchmark_trajectory_requests_performed": 0,
    "qualification_claimed": False,
    "credentials_used": False,
    "customer_data_used": False,
    "external_spend": 0,
}
(EVIDENCE_ROOT / "90_summary.json").write_text(
    canonical_json(summary) + "\n",
    encoding="utf-8",
)

payload_paths = tuple(sorted(EVIDENCE_ROOT.iterdir(), key=lambda path: path.name))
sha_entries = [
    {"path": path.name, "sha256": sha256_file(path), "size_bytes": path.stat().st_size}
    for path in payload_paths
]
(EVIDENCE_ROOT / "99_evidence_sha256.json").write_text(
    canonical_json({"schema_version": "1.0.0", "entries": sha_entries}) + "\n",
    encoding="utf-8",
)

with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVIDENCE_ROOT.iterdir(), key=lambda item: item.name):
        archive.write(path, arcname=path.name)

print(f"artifact={OUTPUT_ZIP}")
print(f"size_bytes={OUTPUT_ZIP.stat().st_size}")
print(f"sha256={sha256_file(OUTPUT_ZIP)}")
print(f"offline_compatibility_status={summary['status']}")
print(f"failed_required_roles={canonical_json(failed)}")
print("model_requests_performed=0")
print("qualification_claimed=false")
print("upload_only_this_file=true")
